# Main (non-linear) -- target shape once `weak_form` is implemented

**This notebook is intentionally NOT runnable yet.** It imports
`surroptim.weak_form.build_residual`, which raises `NotImplementedError` until issue #2's
physics is filled in (`src/surroptim/weak_form/boundary_terms.py` and `residual.py`).

What this notebook DOES show, correctly and runnably as-is: the **wiring** you are
building towards -- `dolfinx.fem.petsc.NonlinearProblem` replacing the direct KSP
linear solve -- and the exact, version-checked API for dolfinx 0.11 (verified against
the pinned `ghcr.io/fenics/dolfinx/dolfinx:v0.11.0` docs, since this API changed
recently: see the note below the solver cell).


In [ ]:
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import mesh, fem
import dolfinx.fem.petsc
from dolfinx.fem.petsc import NonlinearProblem

from surroptim.weak_form import build_residual, build_jacobian


## Geometry, spaces, BCs, boundary tags -- unchanged from the linear stand-in

Reuse `notebooks/guidance/linear_stand_in.ipynb` as-is for this part: same mesh, same
`GAMMA_D` / `GAMMA_N` tagging, same `ds(...)` measure. If the face selection was wrong
there, it will be wrong here too -- that notebook is what you validate first.


In [ ]:
width, thickness = 1.0, 0.5
nx, ny = 60, 20

domain = mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0.0, 0.0]), np.array([width, thickness])],
    [nx, ny],
    cell_type=mesh.CellType.triangle,
)
V = fem.functionspace(domain, ("CG", 1))

def boundary_bottom(x):
    return np.isclose(x[1], 0.0)

def boundary_rest(x):
    return ~np.isclose(x[1], 0.0)

facet_dim = domain.topology.dim - 1
bottom_facets = mesh.locate_entities_boundary(domain, facet_dim, boundary_bottom)
rest_facets = mesh.locate_entities_boundary(domain, facet_dim, boundary_rest)

GAMMA_D, GAMMA_N = 1, 2
facet_indices = np.concatenate([bottom_facets, rest_facets])
facet_markers = np.concatenate([
    np.full_like(bottom_facets, GAMMA_D),
    np.full_like(rest_facets, GAMMA_N),
])
order = np.argsort(facet_indices)
facet_tags = mesh.meshtags(domain, facet_dim, facet_indices[order], facet_markers[order])
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)

dofs_bottom = fem.locate_dofs_geometrical(V, boundary_bottom)
bcs = [fem.dirichletbc(PETSc.ScalarType(0.0), dofs_bottom, V)]

r_weight = fem.Function(V)
r_weight.interpolate(lambda x: np.maximum(np.abs(x[0]), 1e-14))


## Non-linear solve: F/J instead of a/L

Structural change to notice: `T` is now a `fem.Function` directly (the Newton unknown),
**not** a `ufl.TrialFunction` like the legacy `du`. There is no separate trial function
for the residual -- only `build_jacobian` introduces one internally via
`ufl.derivative`.


In [ ]:
T = fem.Function(V)     # the non-linear unknown, iterated in place by SNES
T_n = fem.Function(V)   # known temperature at the previous time step
v = ufl.TestFunction(V)

source = fem.Function(V)  # plug in your TimeGatedGaussian.interpolate(...) per step

dt = 5.0e-3
F = build_residual(
    T, T_n, v, r_weight, dt=dt,
    thermal_capacity=1.0, diffusion_coeff=1.0, advection_coeff=1.0,
    source=source, epsilon=0.8, sigma=5.670374419e-8, t_amb_k=293.15,
    ds=ds(GAMMA_N),  # <- decide here whether Gamma_D also radiates (issue #2, open point)
)
J = build_jacobian(F, T)


## Solver setup -- IMPORTANT dolfinx-0.11-specific note

`dolfinx.nls.petsc.NewtonSolver` (the class named in the original issue text) is
**deprecated** as of dolfinx 0.11, the exact version this project is pinned to.
`dolfinx.fem.petsc.NonlinearProblem` now wraps PETSc SNES *directly* -- there is no
separate Newton solver object to couple it with any more. (The old
`NonlinearProblem` + `NewtonSolver` pairing still exists for backwards compatibility
under the renamed `dolfinx.fem.petsc.NewtonSolverNonlinearProblem`, but do not use
that path here -- it is the legacy one.)

`petsc_options_prefix` is a **mandatory** keyword argument (any unique string is
fine). Convergence tolerances and the linear sub-solve are configured through
`petsc_options`, not through solver attributes:


In [ ]:
problem = NonlinearProblem(
    F, T,
    bcs=bcs,
    J=J,
    petsc_options_prefix="surroptim_radiation_",
    petsc_options={
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
        "snes_linesearch_type": "bt",
        # Conditioning warning from issue #2: sigma ~ 5.67e-8 sits at a WILDLY
        # different scale than the diffusion/mass terms (O(1) here). If SNES
        # stalls or diverges, loosen/tighten these and re-check scaling before
        # assuming the physics (the T**4 term itself) is wrong.
        "snes_rtol": 1e-8,
        "snes_atol": 1e-10,
        "snes_max_it": 50,
    },
)


## Time loop

`T` already holds the previous step's converged solution when `problem.solve()` is
called again, which is exactly the Newton initial guess `T^{n+1,0} = T^n` described in
`physique_formulations.md` -- no explicit reset needed between steps, only `T_n` must
be updated.


In [ ]:
n_steps = 100
for step in range(n_steps):
    T_n.x.array[:] = T.x.array
    # source.interpolate(...)  # update your TimeGatedGaussian at the new time here

    problem.solve()

    reason = problem.solver.getConvergedReason()
    n_it = problem.solver.getIterationNumber()
    assert reason > 0, f"SNES did not converge at step {step} (reason={reason})"
    print(f"step {step}: {n_it} Newton/SNES iterations, reason={reason}")


## Regression check against the linear model (do this before trusting the radiation)

Acceptance criterion from issue #2: with `epsilon=0.0` (or with `T` small enough that
`radiative_flux` is negligible), this non-linear path should reproduce the *existing*
linear `AxisymHeatProblem.solve()` results almost exactly for the first few steps.
Run both side by side with identical parameters before trusting any run where
radiation is actually switched on -- this isolates "did I break the linear case
rewriting it as a residual" from "does the new physics behave correctly".
